In [26]:
# ============================================================
# COLAB CELL — Print 5 random prompts + Gemini 2.5 Pro teacher answers (from TOP-K JSONL)
# ============================================================
import json, os, random
from google.colab import drive

drive.mount("/content/drive")

BASE_DIR = "/content/drive/MyDrive/DL_Local"
TOPK_JSON_PATH   = os.path.join(BASE_DIR, "clinical_pharm_teacher_topk_gemini25pro.jsonl")
PROMPTS_JSON_PATH = os.path.join(BASE_DIR, "clinical_pharm_prompts_10000.jsonl")

assert os.path.exists(TOPK_JSON_PATH), f"❌ File not found: {TOPK_JSON_PATH}"
assert os.path.exists(PROMPTS_JSON_PATH), f"❌ File not found: {PROMPTS_JSON_PATH}"

# -----------------------------
# 1) Load prompt map: id -> prompt
# -----------------------------
id2prompt = {}
with open(PROMPTS_JSON_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        obj = json.loads(line)
        ex_id = obj.get("id")
        if ex_id:
            id2prompt[ex_id] = obj.get("prompt", "")

print(f"✅ Loaded prompts: {len(id2prompt)}")

# -----------------------------
# 2) Stream teacher rows, keep only OK
# -----------------------------
ok_rows = []
with open(TOPK_JSON_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        obj = json.loads(line)
        if obj.get("status") == "ok" and obj.get("teacher_text"):
            ok_rows.append(obj)

print(f"✅ Teacher rows (ok): {len(ok_rows)}")

assert len(ok_rows) >= 5, "❌ Not enough OK rows in teacher file to sample 5."

# -----------------------------
# 3) Sample 5 and print prompt+answer
# -----------------------------
random.seed(43)  # deterministic sample
samples = random.sample(ok_rows, 17)

for i, ex in enumerate(samples, 1):
    ex_id = ex.get("id")
    prompt = id2prompt.get(ex_id, "[PROMPT NOT FOUND for this id]")
    answer = ex.get("teacher_text", "")

    print("\n" + "="*100)
    print(f"SAMPLE {i}")
    print("ID:", ex_id)
    print("Finish reason:", ex.get("finish_reason"))
    print("Usage:", ex.get("usage"))
    print("\n--- PROMPT ---")
    print(prompt[:2500])  # keep readable; raise if you want full
    print("\n--- TEACHER ANSWER (Gemini 2.5 Pro) ---")
    print(answer[:2500])  # keep readable; raise if you want full

print("\n" + "="*100)
print("QUALITY CHECKLIST (use this when reading the 5 samples):")
print("""\
A) Format compliance
   - Decision is exactly "Safe" or "Unsafe"
   - Explanation is 4–8 sentences
   - No extra sections (unless your spec allows e.g. Uncertainty notes)

B) Medical interaction reasoning (high-level)
   - Mentions at least one PK/PD interaction OR explicitly states "no direct interaction" where appropriate
   - Mentions relevant comorbidities (renal/liver function, age) and ties them to risk
   - Avoids hallucinating meds not in the list (e.g., inventing AEDs)

C) Safety posture
   - If "Unsafe": gives a concrete mechanism of harm
   - If "Safe": still notes what to monitor (labs/bleeding/hepatotoxicity) without going off-task
   - No “as an AI” disclaimers, no rambling

D) Internal consistency
   - Reasoning matches the decision
   - No contradictions (e.g., says 'contraindicated' but labels 'Safe')
""")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Loaded prompts: 10000
✅ Teacher rows (ok): 17

SAMPLE 1
ID: ba880477c1e2
Finish reason: FinishReason.STOP
Usage: {'prompt_tokens': 303, 'candidates_tokens': 202, 'thoughts_tokens': 1643, 'total_tokens': 2148}

--- PROMPT ---
QUESTION:
You are a clinical pharmacology decision assistant.
Always base your answers on verified medical knowledge, and avoid speculation.

Patient profile:
- Age: 61
- Sex: Male
- Weight: 92 kg
- Height: 180 cm
- BMI: 28.4
- Renal function: Normal (eGFR ≥ 90)
- Liver function: No known impairment
- Medical history: Generalized anxiety disorder
- Current medications: loratadine, metoprolol
- New medication considered: paracetamol (500 mg every 6 hours as needed, max 3 g/day)

Tasks:
1) Decide if the drug combination is safe for this patient. Answer only "Safe" or "Unsafe".
2) Provide a single explanation (4-8 sentences) that summarize

In [18]:
# ============================================================
# COLAB CELL — Inspect existing TOP-K teacher JSONL
# ============================================================
import json
import os
from google.colab import drive
drive.mount("/content/drive")

TOPK_JSON_PATH = "/content/drive/MyDrive/clinical_pharm_teacher_topk_gemini25pro.jsonl"

if not os.path.exists(TOPK_JSON_PATH):
    raise FileNotFoundError(f"❌ File not found: {TOPK_JSON_PATH}")

rows = []
with open(TOPK_JSON_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

print(f"✅ Total rows found: {len(rows)}")

# ---- Print first example ----
ex = rows[0]
print("\n=== EXAMPLE 0 ===")
print("ID:", ex["id"])
print("Status:", ex["status"])
print("Finish reason:", ex.get("finish_reason"))
print("\n--- Teacher text ---")
print(ex.get("teacher_text", "")[:1000])

# ---- Inspect TOP-K structure ----
steps = ex.get("logprobs_steps", [])
print(f"\nTOP-K steps recorded: {len(steps)}")

if steps:
    print("\n--- First decoding step ---")
    s0 = steps[0]
    print("Chosen token:", s0["chosen"])
    print("Top-K alternatives (first 5):")
    for alt in s0["topk"][:5]:
        print(" ", alt)

# ---- Optional: inspect another random row ----
if len(rows) > 1:
    ex2 = rows[min(10, len(rows)-1)]
    print("\n=== EXAMPLE 10 ===")
    print("ID:", ex2["id"])
    print("Teacher text (head):")
    print(ex2.get("teacher_text", "")[:500])


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Total rows found: 198

=== EXAMPLE 0 ===
ID: aae6f34d220d
Status: ok
Finish reason: FinishReason.STOP

--- Teacher text ---
Decision: Unsafe
Explanation: The primary concern is a significant pharmacodynamic interaction that substantially increases the risk of serious bleeding. The patient is taking apixaban, a potent anticoagulant, for atrial fibrillation. Methotrexate can cause myelosuppression, leading to thrombocytopenia (a low platelet count), which is a key component of blood clotting. The concurrent use of an anticoagulant (apixaban) with a drug that can lower platelet counts (methotrexate) creates a major risk for severe hemorrhage. This risk is further compounded by sertraline, which can impair platelet aggregation, adding a third mechanism that compromises hemostasis. Although the patient's normal renal function aids methotrexate clearance, it does

In [17]:
# ============================================================
# COLAB CELL — Inspect Gemini 3 Pro teacher answers JSONL
# ============================================================
import json, os, random

G3_PATH = "/content/drive/MyDrive/clinical_pharm_teacher_10000__NO_TEST.jsonl"

if not os.path.exists(G3_PATH):
    raise FileNotFoundError(f"❌ File not found: {G3_PATH}")

# Load rows (for big files, this is still fine for a quick check; we can stream if needed)
rows = []
with open(G3_PATH, "r", encoding="utf-8") as f:
    for line in f:
        if line.strip():
            rows.append(json.loads(line))

print(f"✅ Total rows found: {len(rows)}")

# Count statuses
from collections import Counter
cnt = Counter(r.get("status","?") for r in rows)
print("Status counts:", dict(cnt))

# Show 3 random OK examples
ok = [r for r in rows if r.get("status") == "ok"]
print(f"✅ OK rows: {len(ok)}")

for i, ex in enumerate(random.sample(ok, k=min(3, len(ok)))):
    print("\n" + "="*80)
    print(f"EXAMPLE {i+1}")
    print("ID:", ex.get("id"))
    print("used_retry:", ex.get("used_retry"))
    # Gemini3 file schema differs a bit depending on retry/no-retry
    text = ex.get("teacher_output") or ex.get("teacher_text") or ""
    print("Teacher output (head):\n", text[:1200])
    print("\nUsage:", ex.get("usage") or {"first_try": ex.get("first_try",{}).get("usage"), "retry": ex.get("retry",{}).get("usage")})
    print("Finish reason:", ex.get("finish_reason") or ex.get("first_try",{}).get("finish_reason"), "/", ex.get("retry",{}).get("finish_reason"))


✅ Total rows found: 0
Status counts: {}
✅ OK rows: 0


In [12]:
import json, os, random

PATH = "/content/drive/MyDrive/clinical_pharm_teacher_10000__NO_TEST.jsonl"
assert os.path.exists(PATH)

errs = []
with open(PATH, "r", encoding="utf-8") as f:
    for line in f:
        if not line.strip():
            continue
        obj = json.loads(line)
        if obj.get("status") == "error":
            errs.append(obj)
            if len(errs) >= 5:
                break

print(f"Showing {len(errs)} error examples:")
for i,e in enumerate(errs, 1):
    print("\n" + "="*80)
    print("Error example", i)
    print("id:", e.get("id"))
    print("error_stage:", e.get("error_stage"))
    print("error:", e.get("error"))


Showing 0 error examples:


In [16]:
# ============================================================
# COLAB CELL — Rebuild clean Gemini-3 teacher JSONL (OK rows only)
# - Removes thousands of quota "error" rows
# - Keeps first OK per id (dedup)
# ============================================================

import os, json

IN_PATH  = "/content/drive/MyDrive/clinical_pharm_teacher_10000__NO_TEST.jsonl"
OUT_PATH = "/content/drive/MyDrive/clinical_pharm_teacher_10000__NO_TEST.jsonl"

assert os.path.exists(IN_PATH), f"Missing: {IN_PATH}"

kept = 0
dropped = 0
seen = set()

with open(IN_PATH, "r", encoding="utf-8") as fin, open(OUT_PATH, "w", encoding="utf-8") as fout:
    for line in fin:
        if not line.strip():
            continue
        obj = json.loads(line)
        ex_id = obj.get("id")
        if not ex_id or ex_id in seen:
            continue
        seen.add(ex_id)

        if obj.get("status") == "ok":
            fout.write(json.dumps(obj, ensure_ascii=False) + "\n")
            kept += 1
        else:
            dropped += 1

print("✅ Clean file written:", OUT_PATH)
print("kept ok:", kept)
print("dropped non-ok:", dropped)


✅ Clean file written: /content/drive/MyDrive/clinical_pharm_teacher_10000__NO_TEST.jsonl
kept ok: 0
dropped non-ok: 0


In [5]:
# Quick Drive-wide search for the Gemini-3 output
import os

target = "clinical_pharm_teacher_10000.jsonl"
found = False

for root, dirs, files in os.walk("/content/drive"):
    if target in files:
        print("FOUND:", os.path.join(root, target))
        found = True

if not found:
    print("❌ Not found anywhere under /content/drive")


❌ Not found anywhere under /content/drive
